# Project 2 — Boltz-2 co-folding reality check

Run an open **co-folding** foundation model (Boltz-2) on a handful of DHFR +
antifolate pairs, predict the bound complex **and** its affinity, compare to the
experimental ChEMBL values, and ask: **where is the model trustworthy, and where
isn't it?** That last question is the point — it's the 'can you trust the
prediction' theme that this whole portfolio is built around.

**Needs a real GPU (A100/L4).** Use Colab Pro+ or pay-as-you-go A100. A T4 will
be slow/may OOM — keep the complex list short.

## 1. Install Boltz

In [ ]:
!pip -q install boltz
!boltz --help | head -20

## 2. Define a few DHFR–ligand pairs
Pick 3–6 antifolates with known E. coli or human DHFR potency from your cleaned
ChEMBL table (e.g. trimethoprim, methotrexate, plus 1–2 weaker binders), and the
target sequence (wild-type and, ideally, a resistant variant). Boltz takes a YAML
spec per complex.

In [ ]:
import os, textwrap
os.makedirs("boltz_inputs", exist_ok=True)

# Example spec. Replace SEQUENCE with real DHFR (UniProt P0ABQ4) and
# SMILES with each antifolate. One YAML file per complex.
def write_spec(name, sequence, smiles):
    spec = textwrap.dedent(f"""\
    version: 1
    sequences:
      - protein:
          id: A
          sequence: {sequence}
      - ligand:
          id: L
          smiles: '{smiles}'
    properties:
      - affinity:
          binder: L
    """)
    path = f"boltz_inputs/{name}.yaml"
    open(path, "w").write(spec)
    return path

# --- fill these in ---
DHFR_SEQ = "PASTE_E_COLI_DHFR_SEQUENCE"
pairs = {
    "tmp":  "COc1cc(Cc2cnc(N)nc2N)cc(OC)c1OC",     # trimethoprim
    # "mtx": "...",  # methotrexate, etc.
}
specs = [write_spec(n, DHFR_SEQ, smi) for n, smi in pairs.items()]
print(specs)

## 3. Run co-folding + affinity prediction

In [ ]:
# Predict structure + affinity for each complex.
!boltz predict boltz_inputs --use_msa_server --out_dir boltz_out
# outputs: predicted complex structures (.cif) + affinity values per complex

## 4. Collect predictions and compare to experiment

In [ ]:
import json, glob, pandas as pd
rows = []
for f in glob.glob("boltz_out/**/affinity*.json", recursive=True):
    d = json.load(open(f))
    rows.append({"file": f, **d})
pred = pd.DataFrame(rows)
pred  # inspect predicted affinity / binding-probability fields

## 5. The analysis that matters
Don't just report correlation. For the writeup, answer:

1. Does predicted affinity track the experimental ranking of your compounds?
2. Does it capture the **wild-type → resistant-variant** potency shift, or is it
   insensitive to the binding-site mutation?
3. Where it's wrong, *why* — pose confidence low? out of training distribution?

Note the known caveat from the literature that co-folding affinity heads can be
weakly coupled to structural accuracy. A measured, honest verdict here is the
deliverable — it demonstrates exactly the calibration/trust judgement the role
wants, and pairs with the trained model from Project 1 as an independent check.